<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-07-mcp-and-cloud-run/lesson-7.1-fastmcp/notebooks/GCP_Capstone_7.1_FastMCP.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 7.1 Building FastMCP Server — 4 Tools, Streamable HTTP, MCP Inspector
**Netsetos GenAI Engineering — GCP Capstone**

Build a DocuMind MCP server with FastMCP. Define 4 tools. Serve over Streamable HTTP. Test with Inspector and Client.


## Setup


In [ ]:
!pip install -q fastmcp

import fastmcp
print(f'FastMCP version: {fastmcp.__version__}')


## Cell 1: Define the DocuMind MCP Server


In [ ]:
from fastmcp import FastMCP, ToolError
from typing import Literal
from datetime import datetime

mcp = FastMCP('DocuMind')

# Simulated document store
DOCUMENTS = [
    {'id': 'doc-001', 'title': 'Q4 Financial Report', 'category': 'financial',
     'content': 'Revenue increased 15% year-over-year...', 'pages': 24},
    {'id': 'doc-002', 'title': 'Engineering Design Spec', 'category': 'technical',
     'content': 'System architecture uses microservices...', 'pages': 18},
    {'id': 'doc-003', 'title': 'Employee Handbook 2026', 'category': 'hr',
     'content': 'Company policies and procedures...', 'pages': 45},
    {'id': 'doc-004', 'title': 'Marketing Strategy Q2', 'category': 'marketing',
     'content': 'Target audience analysis shows growth...', 'pages': 12},
]
print(f'Document store: {len(DOCUMENTS)} documents')


## Cell 2: Tool 1 — search_documents


In [ ]:
@mcp.tool
def search_documents(query: str, max_results: int = 5) -> list[dict]:
    """Search the document repository for relevant documents.

    Args:
        query: Search query to match against document titles and content.
        max_results: Maximum results to return (default 5, max 20).
    """
    if not query.strip():
        raise ToolError('Search query cannot be empty')
    results = []
    for doc in DOCUMENTS:
        if query.lower() in doc['title'].lower() or query.lower() in doc['content'].lower():
            results.append({'id': doc['id'], 'title': doc['title'],
                            'category': doc['category'], 'relevance': 0.95})
    return results[:min(max_results, 20)]

# Test locally
print(search_documents('financial'))


## Cell 3: Tool 2 — calculate_cost


In [ ]:
@mcp.tool
def calculate_cost(
    page_count: int,
    processing_type: Literal['standard', 'premium', 'enterprise'] = 'standard',
    include_ocr: bool = False
) -> dict:
    """Calculate processing cost for a document.

    Args:
        page_count: Number of pages (must be positive).
        processing_type: Tier — standard ($0.01), premium ($0.03), enterprise ($0.05).
        include_ocr: Add OCR processing at $0.02/page.
    """
    if page_count <= 0:
        raise ToolError('Page count must be positive')
    rates = {'standard': 0.01, 'premium': 0.03, 'enterprise': 0.05}
    ocr = 0.02 if include_ocr else 0.0
    total = round((rates[processing_type] + ocr) * page_count, 2)
    return {'page_count': page_count, 'processing_type': processing_type,
            'total_cost': total, 'currency': 'USD'}

print(calculate_cost(100, 'premium', True))


## Cell 4: Tools 3 & 4 — get_stats + classify_document


In [ ]:
@mcp.tool
def get_stats() -> dict:
    """Get summary statistics about the document repository.

    Returns total documents, pages, and category breakdown.
    """
    total_pages = sum(d['pages'] for d in DOCUMENTS)
    cats = {}
    for d in DOCUMENTS:
        cats[d['category']] = cats.get(d['category'], 0) + 1
    return {'total_documents': len(DOCUMENTS), 'total_pages': total_pages,
            'categories': cats, 'avg_pages': round(total_pages / len(DOCUMENTS), 1)}

@mcp.tool
def classify_document(title: str, content: str) -> dict:
    """Classify a document into a category based on title and content.

    Args:
        title: The document title.
        content: The document content text.
    """
    if not title.strip() or not content.strip():
        raise ToolError('Both title and content are required')
    text = (title + ' ' + content).lower()
    kws = {'financial': ['revenue', 'budget', 'profit'],
           'technical': ['api', 'system', 'architecture'],
           'hr': ['employee', 'policy', 'handbook'],
           'marketing': ['campaign', 'audience', 'brand']}
    scores = {c: sum(1 for k in ws if k in text) for c, ws in kws.items()}
    best = max(scores, key=scores.get)
    return {'category': best, 'confidence': round(min(scores[best]/3, 1.0), 2)}

print(get_stats())
print(classify_document('Budget Report', 'Revenue and profit analysis for Q1'))


## Cell 5: Write the Complete Server File


In [ ]:
server_code = '''
import asyncio, os, logging
from fastmcp import FastMCP, ToolError
from typing import Literal

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

mcp = FastMCP("DocuMind")

DOCUMENTS = [
    {"id": "doc-001", "title": "Q4 Financial Report", "category": "financial",
     "content": "Revenue increased 15% year-over-year...", "pages": 24},
    {"id": "doc-002", "title": "Engineering Design Spec", "category": "technical",
     "content": "System architecture uses microservices...", "pages": 18},
    {"id": "doc-003", "title": "Employee Handbook 2026", "category": "hr",
     "content": "Company policies and procedures...", "pages": 45},
    {"id": "doc-004", "title": "Marketing Strategy Q2", "category": "marketing",
     "content": "Target audience analysis shows growth...", "pages": 12},
]

@mcp.tool
def search_documents(query: str, max_results: int = 5) -> list[dict]:
    """Search the document repository for relevant documents.
    Args:
        query: Search query to match against documents.
        max_results: Max results (default 5, max 20).
    """
    if not query.strip(): raise ToolError("Query cannot be empty")
    results = []
    for doc in DOCUMENTS:
        if query.lower() in doc["title"].lower() or query.lower() in doc["content"].lower():
            results.append({"id": doc["id"], "title": doc["title"],
                            "category": doc["category"], "relevance": 0.95})
    return results[:min(max_results, 20)]

@mcp.tool
def calculate_cost(page_count: int,
                   processing_type: Literal["standard","premium","enterprise"] = "standard",
                   include_ocr: bool = False) -> dict:
    """Calculate processing cost for a document.
    Args:
        page_count: Number of pages (positive).
        processing_type: standard ($0.01), premium ($0.03), enterprise ($0.05).
        include_ocr: Add OCR at $0.02/page.
    """
    if page_count <= 0: raise ToolError("Page count must be positive")
    rates = {"standard": 0.01, "premium": 0.03, "enterprise": 0.05}
    ocr = 0.02 if include_ocr else 0.0
    return {"total_cost": round((rates[processing_type] + ocr) * page_count, 2), "currency": "USD"}

@mcp.tool
def get_stats() -> dict:
    """Get repository statistics: total docs, pages, categories."""
    total_pages = sum(d["pages"] for d in DOCUMENTS)
    cats = {}
    for d in DOCUMENTS: cats[d["category"]] = cats.get(d["category"], 0) + 1
    return {"total_documents": len(DOCUMENTS), "total_pages": total_pages, "categories": cats}

@mcp.tool
def classify_document(title: str, content: str) -> dict:
    """Classify a document into a category.
    Args:
        title: Document title.
        content: Document content text.
    """
    if not title.strip() or not content.strip(): raise ToolError("Both title and content required")
    text = (title + " " + content).lower()
    kws = {"financial":["revenue","budget"],"technical":["api","system"],
           "hr":["employee","policy"],"marketing":["campaign","audience"]}
    scores = {c: sum(1 for k in ws if k in text) for c,ws in kws.items()}
    best = max(scores, key=scores.get)
    return {"category": best, "confidence": round(min(scores[best]/2, 1.0), 2)}

if __name__ == "__main__":
    port = int(os.getenv("PORT", 8000))
    logger.info(f"DocuMind MCP on port {port}")
    asyncio.run(mcp.run_async(transport="streamable-http", host="0.0.0.0", port=port))
'''

with open('documind_server.py', 'w') as f:
    f.write(server_code)
print('Wrote documind_server.py')
print(f'Size: {os.path.getsize("documind_server.py")} bytes')


## Cell 6: Test with FastMCP Client


In [ ]:
# Note: This cell requires the server running in a separate process
# In Colab, you can test locally by importing directly

import asyncio
from fastmcp import Client

async def test_local():
    # Direct connection to the FastMCP server object (no HTTP needed)
    async with Client(mcp) as client:
        # List all tools
        tools = await client.list_tools()
        print(f'=== {len(tools)} Tools Found ===')
        for t in tools:
            print(f'  {t.name}: {t.description[:60]}...')

        # Call each tool
        print('\n=== Tool Calls ===')
        r1 = await client.call_tool('search_documents', {'query': 'financial'})
        print(f'search: {r1}')

        r2 = await client.call_tool('calculate_cost',
                                     {'page_count': 100, 'processing_type': 'enterprise'})
        print(f'cost: {r2}')

        r3 = await client.call_tool('get_stats', {})
        print(f'stats: {r3}')

        r4 = await client.call_tool('classify_document',
                                     {'title': 'Budget Plan', 'content': 'Revenue projections'})
        print(f'classify: {r4}')

await test_local()


## Cell 7: Test Error Handling


In [ ]:
async def test_errors():
    async with Client(mcp) as client:
        # Test ToolError: empty query
        try:
            await client.call_tool('search_documents', {'query': ''})
        except Exception as e:
            print(f'Empty query error: {e}')

        # Test ToolError: negative page count
        try:
            await client.call_tool('calculate_cost', {'page_count': -5})
        except Exception as e:
            print(f'Negative pages error: {e}')

        # Test ToolError: missing content
        try:
            await client.call_tool('classify_document', {'title': 'Test', 'content': ''})
        except Exception as e:
            print(f'Missing content error: {e}')

await test_errors()


## ✅ Lesson 7.1 Complete!

- ✅ MCP protocol: USB-C for AI tools
- ✅ FastMCP server with @mcp.tool decorator
- ✅ 4 DocuMind tools with type hints + docstrings
- ✅ ToolError for user-facing validation errors
- ✅ Streamable HTTP transport (Cloud Run ready)
- ✅ MCP Inspector for visual testing
- ✅ FastMCP Client for programmatic testing
- ✅ curl testing with JSON-RPC protocol

**Next: Lesson 7.2 — Deploy to Cloud Run as a live MCP endpoint**
